# 02b — Silver: Feature Engineering

**Chạy lại mỗi khi thay đổi feature.**

Pipeline:
1. Load `master_df` từ Silver (02a)
2. EDA → `eda_stats`
3. Statistical Inference → `stat_report`
4. Feature Engineering nhóm 1–4
5. Merge → `user_features_df`
6. Validate: đủ 206k user, không null
7. Tách Clustering Vector / Similarity Vector
8. Scale riêng từng vector
9. Upload 5 files
10. Ghi manifest

In [1]:
# Cell 1 — Cài thư viện + clone repo
!pip install supabase -q

import subprocess, sys

subprocess.run([
    "git", "clone",
    "https://github.com/rhyni/retail-recommender-system.git",
    "/kaggle/working/repo"
], check=True)

sys.path.insert(0, "/kaggle/working/repo")
print("✓ Repo đã clone xong")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 6.3 MB/s eta 0:00:00


Cloning into '/kaggle/working/repo'...


✓ Repo đã clone xong


In [2]:
# Cell 2 — Load Secrets + config
from kaggle_secrets import UserSecretsClient
from src import config

secrets = UserSecretsClient()
config.SUPABASE_URL = secrets.get_secret("SUPABASE_URL")
config.SUPABASE_KEY = secrets.get_secret("SUPABASE_KEY")

print("✓ Supabase config loaded")

✓ Supabase config loaded


In [3]:
# Cell 3 — Khai báo RUN_ID dùng chung với 02a
# KHÔNG tạo RUN_ID mới — dùng chung folder với master_df

# SILVER_MASTER_RUN_ID = "20260421T132218Z"  # TODO: copy từ output 02a
# RUN_ID = SILVER_MASTER_RUN_ID              # upload vào cùng folder

# print(f"RUN_ID (dùng chung với 02a): {RUN_ID}")
# print("→ Tất cả files sẽ upload vào cùng folder với master_df")
from datetime import datetime, timezone

RUN_ID               = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
SILVER_MASTER_RUN_ID = "20260424T033356Z"  # chỉ để ĐỌC master_df, không upload

print(f"SILVER_MASTER_RUN_ID   : {SILVER_MASTER_RUN_ID}")
print(f"SILVER_FEATURES RUN_ID : {RUN_ID}")

SILVER_MASTER_RUN_ID   : 20260424T033356Z
SILVER_FEATURES RUN_ID : 20260424T071558Z


---
## PHẦN 1 — Load master_df từ Silver

In [4]:
# Cell 05 — Test Supabase connection
import src.supabase_utils as sutils
from src import config

sutils._client = None
client = sutils.get_client()

print("URL    :", config.SUPABASE_URL)
print("KEY    :", config.SUPABASE_KEY[:20])
print("Bucket :", config.SILVER_BUCKET)

# Test đọc silver-master (để xác nhận master_df có ở đó không)
print(f"\nListing silver-master ({SILVER_MASTER_RUN_ID}):")
res_master = client.storage.from_(config.SILVER_BUCKET).list(
    f"{config.SILVER_MASTER_PREFIX}/{SILVER_MASTER_RUN_ID}"
)
print(f"  OK — {len(res_master)} objects")
print(f"  {[x['name'] for x in res_master[:5]]}")

URL    : https://pqcjemgkbhkpcopfytqz.supabase.co
KEY    : sb_publishable_NnymO
Bucket : silver-data

Listing silver-master (20260424T033356Z):
  OK — 2 objects
  ['_manifest.json', 'master_df']


In [5]:
# Cell 4 — Load master_df
from src.io_utils import load_dataframe

print("Loading master_df from Silver...")
master_df = load_dataframe(
    config.SILVER_BUCKET,
    config.SILVER_MASTER_PREFIX,  # ← đọc từ silver-master
    SILVER_MASTER_RUN_ID,
    "master_df",
)

print(f"\n✓ master_df : {len(master_df):,} rows | {len(master_df.columns)} cols")
print(f"  Columns   : {list(master_df.columns)}")
master_df.dtypes

Loading master_df from Silver...
  ✓ loaded [master_df]: 65 chunks | 32,434,489 rows

✓ master_df : 32,434,489 rows | 16 cols
  Columns   : ['order_id', 'product_id', 'add_to_cart_order', 'reordered', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order', 'product_name', 'aisle_id', 'department_id', 'aisle', 'department', 'is_first_order']


order_id                     int32
product_id                   int32
add_to_cart_order            int16
reordered                     int8
user_id                      int32
eval_set                  category
order_number                 int16
order_dow                     int8
order_hour_of_day             int8
days_since_prior_order     float32
product_name                object
aisle_id                     int16
department_id                 int8
aisle                       object
department                  object
is_first_order                int8
dtype: object

---
## PHẦN 2 — EDA → eda_stats

In [6]:
# Cell 6 — EDA: thống kê tổng quan
import pandas as pd
import numpy as np

print("EDA — TỔNG QUAN")

# ── orders chỉ lấy 1 dòng/order để không bị double-count ──
orders_unique = master_df.drop_duplicates(subset='order_id')

eda_stats = {
    # ── Cơ bản ───────────────────────────────────────────
    "n_users":           int(master_df['user_id'].nunique()),
    "n_orders":          int(master_df['order_id'].nunique()),
    "n_products":        int(master_df['product_id'].nunique()),
    "n_aisles":          int(master_df['aisle_id'].nunique()),
    "n_departments":     int(master_df['department_id'].nunique()),
    "total_line_items":  int(len(master_df)),

    # ── Đơn hàng per user ────────────────────────────────
    "orders_per_user_mean":   float(orders_unique.groupby('user_id')['order_id'].count().mean()),
    "orders_per_user_median": float(orders_unique.groupby('user_id')['order_id'].count().median()),
    "orders_per_user_max":    int(orders_unique.groupby('user_id')['order_id'].count().max()),

    # ── Items per order ───────────────────────────────────
    "items_per_order_mean":   float(master_df.groupby('order_id')['product_id'].count().mean()),
    "items_per_order_median": float(master_df.groupby('order_id')['product_id'].count().median()),
    "items_per_order_max":    int(master_df.groupby('order_id')['product_id'].count().max()),

    # ── Reorder ───────────────────────────────────────────
    "reorder_rate_global":    float(master_df['reordered'].mean()),

    # ── Thời gian ─────────────────────────────────────────
    "days_since_prior_mean":  float(
        master_df[master_df['is_first_order'] == 0]['days_since_prior_order'].mean()
    ),
    "peak_hour":  int(orders_unique['order_hour_of_day'].value_counts().idxmax()),
    "peak_dow":   int(orders_unique['order_dow'].value_counts().idxmax()),
}

for k, v in eda_stats.items():
    print(f"  {k:<35}: {v}")

print("\n✅ eda_stats tính xong")

EDA — TỔNG QUAN
  n_users                            : 206209
  n_orders                           : 3214874
  n_products                         : 49677
  n_aisles                           : 134
  n_departments                      : 21
  total_line_items                   : 32434489
  orders_per_user_mean               : 15.590367054784224
  orders_per_user_median             : 9.0
  orders_per_user_max                : 99
  items_per_order_mean               : 10.088883421247614
  items_per_order_median             : 8.0
  items_per_order_max                : 145
  reorder_rate_global                : 0.5896974667922161
  days_since_prior_mean              : 11.104089736938477
  peak_hour                          : 10
  peak_dow                           : 0

✅ eda_stats tính xong


In [7]:
# Cell 7 — EDA: phân phối theo department + aisle top 10
print("Top 10 department theo số lượt mua:")
dept_counts = (
    master_df.groupby('department')['product_id']
    .count()
    .sort_values(ascending=False)
    .head(10)
)
print(dept_counts.to_string())

print("\nTop 10 aisle theo số lượt mua:")
aisle_counts = (
    master_df.groupby('aisle')['product_id']
    .count()
    .sort_values(ascending=False)
    .head(10)
)
print(aisle_counts.to_string())

# Lưu vào eda_stats
eda_stats['top10_departments'] = dept_counts.index.tolist()
eda_stats['top10_aisles']      = aisle_counts.index.tolist()

Top 10 department theo số lượt mua:
department
produce            9479291
dairy eggs         5414016
snacks             2887550
beverages          2690129
frozen             2236432
pantry             1875577
bakery             1176787
canned goods       1068058
deli               1051249
dry goods pasta     866627

Top 10 aisle theo số lượt mua:
aisle
fresh fruits                     3642188
fresh vegetables                 3418021
packaged vegetables fruits       1765313
yogurt                           1452343
packaged cheese                   979763
milk                              891015
water seltzer sparkling water     841533
chips pretzels                    722470
soy lactosefree                   638253
bread                             584834


In [8]:
# Cell 8 — Convert eda_stats → DataFrame để upload
eda_stats_df = pd.DataFrame([
    {"metric": k, "value": str(v)}
    for k, v in eda_stats.items()
])

print(f"✓ eda_stats_df: {len(eda_stats_df)} rows")
eda_stats_df

✓ eda_stats_df: 18 rows


,metric,value
0,n_users,206209
1,n_orders,3214874
2,n_products,49677
3,n_aisles,134
4,n_departments,21
5,total_line_items,32434489
6,orders_per_user_mean,15.590367054784224
7,orders_per_user_median,9.0
8,orders_per_user_max,99
9,items_per_order_mean,10.088883421247614


---
## PHẦN 3 — Statistical Inference → stat_report

In [9]:
# Cell 9 — Statistical Inference
from scipy import stats

print("STATISTICAL INFERENCE")

stat_records = []

# ── 1. T-test: items/order cho user mua ≥10 vs < 10 đơn ──
order_counts  = master_df.groupby('user_id')['order_id'].nunique()
heavy_users   = order_counts[order_counts >= 10].index
light_users   = order_counts[order_counts < 10].index

items_heavy = master_df[master_df['user_id'].isin(heavy_users)] \
    .groupby('order_id')['product_id'].count().values
items_light = master_df[master_df['user_id'].isin(light_users)] \
    .groupby('order_id')['product_id'].count().values

t_stat, p_val = stats.ttest_ind(items_heavy, items_light, equal_var=False)
stat_records.append({
    "test":        "ttest_items_per_order_heavy_vs_light",
    "statistic":   round(float(t_stat), 4),
    "p_value":     round(float(p_val), 6),
    "significant": bool(p_val < 0.05),
    "note":        "heavy>=10 orders vs light<10 orders",
})
print(f"  T-test items/order heavy vs light : t={t_stat:.4f} | p={p_val:.6f}")

# ── 2. T-test: reorder_rate user order_dow weekday vs weekend ──
orders_dedup = master_df.drop_duplicates(subset='order_id')[['order_id', 'user_id', 'order_dow']].copy()
reorder_per_order = master_df.groupby('order_id')['reordered'].mean().rename('reorder_rate')
orders_dedup = orders_dedup.join(reorder_per_order, on='order_id')

weekend_orders = orders_dedup[orders_dedup['order_dow'].isin([0, 1])]['reorder_rate'].dropna()
weekday_orders = orders_dedup[~orders_dedup['order_dow'].isin([0, 1])]['reorder_rate'].dropna()

t2, p2 = stats.ttest_ind(weekend_orders, weekday_orders, equal_var=False)
stat_records.append({
    "test":        "ttest_reorder_rate_weekend_vs_weekday",
    "statistic":   round(float(t2), 4),
    "p_value":     round(float(p2), 6),
    "significant": bool(p2 < 0.05),
    "note":        "dow 0-1 (weekend) vs 2-6 (weekday)",
})
print(f"  T-test reorder rate weekend vs weekday: t={t2:.4f} | p={p2:.6f}")

# ── 3. Chi-square: order_dow vs peak_hour (sáng/chiều/tối) ──
orders_dedup['time_slot'] = pd.cut(
    orders_dedup['order_hour_of_day'] if 'order_hour_of_day' in orders_dedup.columns
    else master_df.drop_duplicates('order_id').set_index('order_id')['order_hour_of_day'].reindex(orders_dedup['order_id']).values,
    bins=[0, 6, 12, 18, 24],
    labels=['night', 'morning', 'afternoon', 'evening'],
    right=False,
)

# Nếu cột chưa có, join từ master
if 'time_slot' not in orders_dedup.columns or orders_dedup['time_slot'].isnull().all():
    hour_map = master_df.drop_duplicates('order_id').set_index('order_id')['order_hour_of_day']
    orders_dedup['order_hour_of_day'] = orders_dedup['order_id'].map(hour_map)
    orders_dedup['time_slot'] = pd.cut(
        orders_dedup['order_hour_of_day'],
        bins=[0, 6, 12, 18, 24], labels=['night', 'morning', 'afternoon', 'evening'], right=False
    )

contingency = pd.crosstab(orders_dedup['order_dow'], orders_dedup['time_slot'])
chi2, p3, dof, expected = stats.chi2_contingency(contingency)
stat_records.append({
    "test":        "chi2_order_dow_vs_time_slot",
    "statistic":   round(float(chi2), 4),
    "p_value":     round(float(p3), 6),
    "significant": bool(p3 < 0.05),
    "note":        f"dof={dof}; time_slot: night/morning/afternoon/evening",
})
print(f"  Chi2 order_dow vs time_slot : chi2={chi2:.4f} | p={p3:.6f} | dof={dof}")

# ── 4. Pearson: n_orders vs reorder_rate per user ──
user_order_count   = master_df.groupby('user_id')['order_id'].nunique()
user_reorder_rate  = master_df.groupby('user_id')['reordered'].mean()
corr_df = pd.DataFrame({'n_orders': user_order_count, 'reorder_rate': user_reorder_rate}).dropna()

r, p4 = stats.pearsonr(corr_df['n_orders'], corr_df['reorder_rate'])
stat_records.append({
    "test":        "pearsonr_n_orders_vs_reorder_rate",
    "statistic":   round(float(r), 4),
    "p_value":     round(float(p4), 6),
    "significant": bool(p4 < 0.05),
    "note":        "per-user: total orders vs reorder rate",
})
print(f"  Pearson n_orders vs reorder_rate: r={r:.4f} | p={p4:.6f}")

stat_report_df = pd.DataFrame(stat_records)
print(f"\n✅ stat_report_df: {len(stat_report_df)} tests")
stat_report_df

STATISTICAL INFERENCE
  T-test items/order heavy vs light : t=32.1099 | p=0.000000
  T-test reorder rate weekend vs weekday: t=16.7241 | p=0.000000
  Chi2 order_dow vs time_slot : chi2=8044.7601 | p=0.000000 | dof=18
  Pearson n_orders vs reorder_rate: r=0.6369 | p=0.000000

✅ stat_report_df: 4 tests


,test,statistic,p_value,significant,note
0,ttest_items_per_order_heavy_vs_light,32.1099,0.0,True,heavy>=10 orders vs light<10 orders
1,ttest_reorder_rate_weekend_vs_weekday,16.7241,0.0,True,dow 0-1 (weekend) vs 2-6 (weekday)
2,chi2_order_dow_vs_time_slot,8044.7601,0.0,True,dof=18; time_slot: night/morning/afternoon/eve...
3,pearsonr_n_orders_vs_reorder_rate,0.6369,0.0,True,per-user: total orders vs reorder rate


In [10]:
# Cell 9b — Co-occurrence Chi2 Test: top 20 cặp department
import itertools
from scipy.stats import chi2_contingency

print("CO-OCCURRENCE CHI2 TEST — Department pairs")

# Mỗi đơn hàng có những department nào
order_dept = (
    master_df[['order_id', 'department']]
    .drop_duplicates()
)
total_orders = order_dept['order_id'].nunique()
dept_list    = order_dept['department'].unique()

print(f"  Total orders    : {total_orders:,}")
print(f"  Total dept pairs: {len(list(itertools.combinations(dept_list, 2)))}")

cooc_results = []

for dept_a, dept_b in itertools.combinations(dept_list, 2):
    orders_a = set(order_dept[order_dept['department'] == dept_a]['order_id'])
    orders_b = set(order_dept[order_dept['department'] == dept_b]['order_id'])

    both    = len(orders_a & orders_b)
    only_a  = len(orders_a) - both
    only_b  = len(orders_b) - both
    neither = total_orders - len(orders_a | orders_b)

    # Bỏ qua nếu bảng có cell = 0 → chi2 không hợp lệ
    if min(both, only_a, only_b, neither) == 0:
        continue

    chi2, p, _, _ = chi2_contingency([[both, only_a], [only_b, neither]])

    # Lift = P(A∩B) / (P(A) * P(B))
    lift = (both / total_orders) / ((len(orders_a) / total_orders) * (len(orders_b) / total_orders))

    cooc_results.append({
        'dept_a':             dept_a,
        'dept_b':             dept_b,
        'chi2':               round(float(chi2), 4),
        'p_value':            round(float(p), 8),
        'lift':               round(float(lift), 4),
        'co_occurrence_count': both,
        'significant':        bool(p < 0.05),
    })

cooc_df = (
    pd.DataFrame(cooc_results)
    .query("significant == True")
    .sort_values('lift', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print(f"\n✅ Top 20 cặp department có p-value < 0.05:")
print(cooc_df[['dept_a', 'dept_b', 'chi2', 'p_value', 'lift', 'co_occurrence_count']])

# Append vào stat_report_df hiện có
stat_report_df = pd.concat([stat_report_df, cooc_df], ignore_index=True)
print(f"\n✅ stat_report_df tổng: {len(stat_report_df)} dòng")

CO-OCCURRENCE CHI2 TEST — Department pairs
  Total orders    : 3,214,874
  Total dept pairs: 210

✅ Top 20 cặp department có p-value < 0.05:
             dept_a           dept_b         chi2  p_value    lift  \
0           alcohol            other    4227.6283      0.0  3.1002   
1              pets          alcohol    3791.5991      0.0  2.5236   
2         household             pets   23693.3134      0.0  2.5122   
3     personal care        household   96802.4194      0.0  2.2632   
4     personal care             pets   10131.1127      0.0  2.2351   
5     personal care            other    4785.9066      0.0  2.1082   
6     international     canned goods   54353.1730      0.0  1.9217   
7           missing            other     536.0345      0.0  1.8965   
8   dry goods pasta     canned goods  140712.0495      0.0  1.8441   
9     international  dry goods pasta   34199.2157      0.0  1.7932   
10             pets            other     362.6506      0.0  1.7388   
11        household

---
## PHẦN 4 — Feature Engineering

In [11]:
# Cell 10 — Nhóm 1: Activity Features
print("── Nhóm 1: Activity Features ──")

non_first = master_df[master_df['is_first_order'] == 0].copy()

# ── avg_basket_size: tính riêng ngoài agg để tránh sai logic ──
basket_per_order = (
    master_df
    .groupby('order_id')['product_id']
    .count()
    .rename('basket_size')
)
order_user_map = (
    master_df
    .drop_duplicates(subset='order_id')
    .set_index('order_id')['user_id']
)
avg_basket = (
    basket_per_order
    .groupby(order_user_map)
    .mean()
    .rename('avg_basket_size')
    .reset_index()
)

# ── Các feature còn lại ──
group1_base = master_df.groupby('user_id').agg(
    total_orders        = ('order_id',    'nunique'),
    total_items_bought  = ('product_id',  'count'),
    total_unique_items  = ('product_id',  'nunique'),
    reorder_rate        = ('reordered',   'mean'),
).reset_index()

# avg_days_between_orders — chỉ tính trên non-first-order
avg_days = (
    non_first
    .drop_duplicates(subset='order_id')
    .groupby('user_id')['days_since_prior_order']
    .mean()
    .rename('avg_days_between_orders')
    .reset_index()
)

# pct_days_capped — tỷ lệ đơn bị cap tại 30 ngày
pct_capped = (
    non_first
    .drop_duplicates(subset='order_id')
    .groupby('user_id')['days_since_prior_order']
    .apply(lambda x: (x == 30).sum() / len(x))
    .rename('pct_days_capped')
    .reset_index()
)

# purchase_regularity = 1 - CV (đo độ đều đặn)
interval_std = (
    non_first
    .drop_duplicates(subset='order_id')
    .groupby('user_id')['days_since_prior_order']
    .std()
    .rename('interval_std')
    .reset_index()
)

# Merge tất cả
group1 = (
    group1_base
    .merge(avg_basket,    on='user_id', how='left')
    .merge(avg_days,      on='user_id', how='left')
    .merge(pct_capped,    on='user_id', how='left')
    .merge(interval_std,  on='user_id', how='left')
)

# purchase_regularity sau khi có avg_days và interval_std
group1['purchase_regularity'] = (
    1 - (group1['interval_std'] / (group1['avg_days_between_orders'] + 1e-9))
).clip(0, 1).astype('float32')
group1 = group1.drop(columns=['interval_std'])

# items_per_order
group1['items_per_order'] = (
    group1['total_items_bought'] / group1['total_orders']
).astype('float32')

# Ép kiểu
group1['total_orders']            = group1['total_orders'].astype('int16')
group1['total_items_bought']      = group1['total_items_bought'].astype('int32')
group1['total_unique_items']      = group1['total_unique_items'].astype('int32')
group1['reorder_rate']            = group1['reorder_rate'].astype('float32')
group1['avg_basket_size']         = group1['avg_basket_size'].astype('float32')
group1['avg_days_between_orders'] = group1['avg_days_between_orders'].astype('float32')
group1['pct_days_capped']         = group1['pct_days_capped'].astype('float32')

print(f"✓ group1 : {len(group1):,} users | {len(group1.columns)} cols")
print(f"  Cols   : {list(group1.columns)}")
group1.head(3)

── Nhóm 1: Activity Features ──
✓ group1 : 206,209 users | 10 cols
  Cols   : ['user_id', 'total_orders', 'total_items_bought', 'total_unique_items', 'reorder_rate', 'avg_basket_size', 'avg_days_between_orders', 'pct_days_capped', 'purchase_regularity', 'items_per_order']


,user_id,total_orders,total_items_bought,total_unique_items,reorder_rate,avg_basket_size,avg_days_between_orders,pct_days_capped,purchase_regularity,items_per_order
0,1,10,59,18,0.694915,5.900000,19.555555,0.111111,0.519542,5.900000
1,2,14,195,102,0.476923,13.928572,15.230769,0.153846,0.352162,13.928572
2,3,12,88,33,0.625000,7.333333,12.090909,0.000000,0.555449,7.333333


In [12]:
# Cell 11 — Nhóm 2: Temporal Features (thói quen thời gian)
print("── Nhóm 2: Temporal Features ──")

orders_dedup_full = (
    master_df
    .drop_duplicates(subset='order_id')[[
        'user_id', 'order_id', 'order_dow',
        'order_hour_of_day', 'order_number',
    ]]
    .copy()
)

# Entropy của order_dow (đa dạng ngày trong tuần)
def dow_entropy(series):
    counts = series.value_counts(normalize=True)
    return float(-(counts * np.log2(counts + 1e-9)).sum())

group2 = orders_dedup_full.groupby('user_id').agg(
    preferred_dow         = ('order_dow',          lambda x: int(x.mode()[0])),
    preferred_hour        = ('order_hour_of_day',  lambda x: int(x.mode()[0])),
    dow_entropy           = ('order_dow',          dow_entropy),
    pct_weekend_orders    = ('order_dow',          lambda x: float((x.isin([0, 1])).mean())),
    pct_morning_orders    = ('order_hour_of_day',  lambda x: float(((x >= 6) & (x < 12)).mean())),
    pct_evening_orders    = ('order_hour_of_day',  lambda x: float(((x >= 18) & (x < 24)).mean())),
).reset_index()

group2['preferred_dow']      = group2['preferred_dow'].astype('int8')
group2['preferred_hour']     = group2['preferred_hour'].astype('int8')
group2['dow_entropy']        = group2['dow_entropy'].astype('float32')
group2['pct_weekend_orders'] = group2['pct_weekend_orders'].astype('float32')
group2['pct_morning_orders'] = group2['pct_morning_orders'].astype('float32')
group2['pct_evening_orders'] = group2['pct_evening_orders'].astype('float32')

print(f"✓ group2 : {len(group2):,} users | {len(group2.columns)} cols")
print(f"  Cols   : {list(group2.columns)}")
group2.head(3)

── Nhóm 2: Temporal Features ──
✓ group2 : 206,209 users | 7 cols
  Cols   : ['user_id', 'preferred_dow', 'preferred_hour', 'dow_entropy', 'pct_weekend_orders', 'pct_morning_orders', 'pct_evening_orders']


,user_id,preferred_dow,preferred_hour,dow_entropy,pct_weekend_orders,pct_morning_orders,pct_evening_orders
0,1,1,7,1.970951,0.300000,0.600000,0.00
1,2,1,10,2.005978,0.357143,0.857143,0.00
2,3,0,16,1.729574,0.666667,0.000000,0.25


In [13]:
# Cell 12 — Nhóm 3: Product Preference Features (sở thích sản phẩm)
print("── Nhóm 3: Product Preference Features ──")

# Top department per user
user_dept = (
    master_df
    .groupby(['user_id', 'department_id'])['product_id']
    .count()
    .reset_index(name='dept_count')
)
top_dept = (
    user_dept
    .sort_values('dept_count', ascending=False)
    .drop_duplicates(subset='user_id')
    .rename(columns={'department_id': 'top_department_id'})[['user_id', 'top_department_id']]
)

# Top aisle per user
user_aisle = (
    master_df
    .groupby(['user_id', 'aisle_id'])['product_id']
    .count()
    .reset_index(name='aisle_count')
)
top_aisle = (
    user_aisle
    .sort_values('aisle_count', ascending=False)
    .drop_duplicates(subset='user_id')
    .rename(columns={'aisle_id': 'top_aisle_id'})[['user_id', 'top_aisle_id']]
)

# Breadth: số department khác nhau
dept_breadth = (
    master_df
    .groupby('user_id')['department_id']
    .nunique()
    .rename('n_departments_bought')
    .reset_index()
)

# Organic preference: tỉ lệ sản phẩm có 'organic' trong tên
master_df['is_organic'] = master_df['product_name'].str.lower().str.contains('organic').astype('int8')
organic_pct = (
    master_df
    .groupby('user_id')['is_organic']
    .mean()
    .rename('pct_organic')
    .reset_index()
)

# Merge nhóm 3
group3 = top_dept.merge(top_aisle, on='user_id', how='left') \
                 .merge(dept_breadth, on='user_id', how='left') \
                 .merge(organic_pct, on='user_id', how='left')

group3['top_department_id']    = group3['top_department_id'].astype('int8')
group3['top_aisle_id']         = group3['top_aisle_id'].astype('int16')
group3['n_departments_bought'] = group3['n_departments_bought'].astype('int8')
group3['pct_organic']          = group3['pct_organic'].astype('float32')

print(f"✓ group3 : {len(group3):,} users | {len(group3.columns)} cols")
print(f"  Cols   : {list(group3.columns)}")
group3.head(3)

── Nhóm 3: Product Preference Features ──
✓ group3 : 206,209 users | 5 cols
  Cols   : ['user_id', 'top_department_id', 'top_aisle_id', 'n_departments_bought', 'pct_organic']


,user_id,top_department_id,top_aisle_id,n_departments_bought,pct_organic
0,75124,16,120,6,0.009864
1,33731,16,120,13,0.034280
2,39901,16,120,6,0.483820


In [14]:
# Cell 13 — Nhóm 4: Loyalty & Recency Features
print("── Nhóm 4: Loyalty & Recency Features ──")

# max_order_number = order cuối cùng (proxy cho recency / độ trung thành)
group4_base = (
    master_df
    .drop_duplicates(subset='order_id')
    .groupby('user_id')
    .agg(
        max_order_number     = ('order_number', 'max'),
        last_order_dow       = ('order_dow',    'last'),
        last_order_hour      = ('order_hour_of_day', 'last'),
    )
    .reset_index()
)

# Tỉ lệ sản phẩm mua lần đầu (add_to_cart_order == 1)
first_slot_rate = (
    master_df
    .groupby('user_id')
    .apply(lambda df: (df['add_to_cart_order'] == 1).mean())
    .rename('pct_first_slot')
    .reset_index()
)

# Tỉ lệ đơn hàng thỉnh thoảng (days > 14) trong lần mua
infrequent_rate = (
    non_first
    .drop_duplicates(subset='order_id')
    .groupby('user_id')
    .apply(lambda df: (df['days_since_prior_order'] > 14).mean())
    .rename('pct_infrequent_orders')
    .reset_index()
)

group4 = group4_base.merge(first_slot_rate, on='user_id', how='left') \
                    .merge(infrequent_rate, on='user_id', how='left')

group4['max_order_number']     = group4['max_order_number'].astype('int16')
group4['last_order_dow']       = group4['last_order_dow'].astype('int8')
group4['last_order_hour']      = group4['last_order_hour'].astype('int8')
group4['pct_first_slot']       = group4['pct_first_slot'].astype('float32')
group4['pct_infrequent_orders']= group4['pct_infrequent_orders'].astype('float32')

print(f"✓ group4 : {len(group4):,} users | {len(group4.columns)} cols")
print(f"  Cols   : {list(group4.columns)}")
group4.head(3)

── Nhóm 4: Loyalty & Recency Features ──


/tmp/ipykernel_16/3945047065.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: (df['add_to_cart_order'] == 1).mean())


✓ group4 : 206,209 users | 6 cols
  Cols   : ['user_id', 'max_order_number', 'last_order_dow', 'last_order_hour', 'pct_first_slot', 'pct_infrequent_orders']


/tmp/ipykernel_16/3945047065.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: (df['days_since_prior_order'] > 14).mean())


,user_id,max_order_number,last_order_dow,last_order_hour,pct_first_slot,pct_infrequent_orders
0,1,10,2,7,0.169492,0.777778
1,2,14,4,11,0.071795,0.307692
2,3,12,0,17,0.136364,0.363636


---
## PHẦN 5 — Merge → user_features_df + Validate

In [15]:
# Cell 14 — Merge 4 nhóm → user_features_df
print("Merging 4 groups...")

user_features_df = (
    group1
    .merge(group2, on='user_id', how='inner')
    .merge(group3, on='user_id', how='inner')
    .merge(group4, on='user_id', how='inner')
)

print(f"✓ user_features_df: {len(user_features_df):,} users | {len(user_features_df.columns)} cols")
print(f"  Cols: {list(user_features_df.columns)}")
user_features_df.head(3)

Merging 4 groups...
✓ user_features_df: 206,209 users | 25 cols
  Cols: ['user_id', 'total_orders', 'total_items_bought', 'total_unique_items', 'reorder_rate', 'avg_basket_size', 'avg_days_between_orders', 'pct_days_capped', 'purchase_regularity', 'items_per_order', 'preferred_dow', 'preferred_hour', 'dow_entropy', 'pct_weekend_orders', 'pct_morning_orders', 'pct_evening_orders', 'top_department_id', 'top_aisle_id', 'n_departments_bought', 'pct_organic', 'max_order_number', 'last_order_dow', 'last_order_hour', 'pct_first_slot', 'pct_infrequent_orders']


,user_id,total_orders,total_items_bought,total_unique_items,reorder_rate,avg_basket_size,avg_days_between_orders,pct_days_capped,purchase_regularity,items_per_order,...,pct_evening_orders,top_department_id,top_aisle_id,n_departments_bought,pct_organic,max_order_number,last_order_dow,last_order_hour,pct_first_slot,pct_infrequent_orders
0,1,10,59,18,0.694915,5.900000,19.555555,0.111111,0.519542,5.900000,...,0.00,19,77,7,0.254237,10,2,7,0.169492,0.777778
1,2,14,195,102,0.476923,13.928572,15.230769,0.153846,0.352162,13.928572,...,0.00,16,120,13,0.256410,14,4,11,0.071795,0.307692
2,3,12,88,33,0.625000,7.333333,12.090909,0.000000,0.555449,7.333333,...,0.25,4,24,9,0.340909,12,0,17,0.136364,0.363636


In [16]:
# Cell 15 — Validate user_features_df
print("Validating user_features_df...")

# 1. Đủ ~206k users
n_users = len(user_features_df)
assert 206_000 <= n_users <= 207_000, f"❌ n_users={n_users:,} ngoài kỳ vọng"
print(f"  ✓ n_users: {n_users:,}")

# 2. Không null
null_summary = user_features_df.isnull().sum()
null_cols    = null_summary[null_summary > 0]

if len(null_cols) > 0:
    print(f"  ⚠️  Cột có null (sẽ fillna):")
    print(null_cols)
    # fillna cho float: median; cho int: mode
    for col in null_cols.index:
        if user_features_df[col].dtype in ['float32', 'float64']:
            user_features_df[col] = user_features_df[col].fillna(
                user_features_df[col].median()
            ).astype('float32')
        else:
            user_features_df[col] = user_features_df[col].fillna(
                user_features_df[col].mode()[0]
            )
    print("  → fillna xong")
else:
    print("  ✓ Không có null")

# 3. Không duplicate user_id
dup = user_features_df['user_id'].duplicated().sum()
assert dup == 0, f"❌ Duplicate user_id: {dup}"
print(f"  ✓ Không duplicate user_id")

# 4. Memory check
mem_mb = user_features_df.memory_usage(deep=True).sum() / 1e6
print(f"  ✓ Memory: {mem_mb:.1f} MB")

print("\n✅ user_features_df hợp lệ")

Validating user_features_df...
  ✓ n_users: 206,209
  ✓ Không có null
  ✓ Không duplicate user_id
  ✓ Memory: 15.7 MB

✅ user_features_df hợp lệ


---
## PHẦN 6 — Tách Vector + Scale

In [17]:
# Cell 16 — Định nghĩa Clustering Vector vs Similarity Vector

# ── Clustering Vector ─────────────────────────────────────────────
# Dùng cho K-Means / HDBSCAN để phân cụm hành vi tổng thể
# Chọn các feature liên tục, không phụ thuộc catalog (không có ID)
CLUSTERING_COLS = [
    'user_id',
    # Activity
    'total_orders',
    'avg_basket_size',
    'reorder_rate',
    'items_per_order',
    'avg_days_between_orders',
    # Temporal
    'dow_entropy',
    'pct_weekend_orders',
    'pct_morning_orders',
    'pct_evening_orders',
    # Product breadth
    'n_departments_bought',
    'pct_organic',
    # Loyalty
    'max_order_number',
    'pct_first_slot',
    'pct_infrequent_orders',
]

# ── Similarity Vector ─────────────────────────────────────────────
# Dùng cho Collaborative Filtering / cosine similarity
# Bao gồm cả preference ID (top dept / aisle) vì phản ánh "sở thích"
SIMILARITY_COLS = [
    'user_id',
    # Activity
    'total_orders',
    'total_unique_items',
    'reorder_rate',
    'avg_basket_size',
    # Temporal
    'preferred_dow',
    'preferred_hour',
    'pct_weekend_orders',
    # Product preference
    'top_department_id',
    'top_aisle_id',
    'n_departments_bought',
    'pct_organic',
    # Loyalty
    'max_order_number',
    'pct_first_slot',
]

print(f"Clustering cols : {len(CLUSTERING_COLS)-1} features")
print(f"Similarity cols : {len(SIMILARITY_COLS)-1} features")

Clustering cols : 14 features
Similarity cols : 13 features


In [18]:
# Cell 17 — Tách ra 2 DataFrame vector (chưa scale)

clustering_raw_df = user_features_df[CLUSTERING_COLS].copy()
similarity_raw_df = user_features_df[SIMILARITY_COLS].copy()

print(f"✓ clustering_raw_df : {clustering_raw_df.shape}")
print(f"✓ similarity_raw_df : {similarity_raw_df.shape}")

✓ clustering_raw_df : (206209, 15)
✓ similarity_raw_df : (206209, 14)


In [19]:
# Cell 18 — Scale Clustering Vector: StandardScaler (zero-mean, unit-variance)
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

clust_feature_cols = [c for c in CLUSTERING_COLS if c != 'user_id']

scaler_clust = StandardScaler()
clust_scaled = scaler_clust.fit_transform(clustering_raw_df[clust_feature_cols])

clustering_scaled_df = pd.DataFrame(
    clust_scaled,
    columns=[f"{c}_scaled" for c in clust_feature_cols],
)
clustering_scaled_df.insert(0, 'user_id', clustering_raw_df['user_id'].values)

# Ép float32 để tiết kiệm RAM
for col in clustering_scaled_df.columns:
    if col != 'user_id':
        clustering_scaled_df[col] = clustering_scaled_df[col].astype('float32')

print(f"✓ clustering_scaled_df : {clustering_scaled_df.shape}")
print(f"  mean ≈ 0 check  : {clustering_scaled_df.drop('user_id', axis=1).mean().abs().max():.4f} (kỳ vọng < 0.01)")
print(f"  std  ≈ 1 check  : {clustering_scaled_df.drop('user_id', axis=1).std().mean():.4f} (kỳ vọng ≈ 1)")
clustering_scaled_df.head(3)

✓ clustering_scaled_df : (206209, 15)
  mean ≈ 0 check  : 0.0000 (kỳ vọng < 0.01)
  std  ≈ 1 check  : 1.0000 (kỳ vọng ≈ 1)


,user_id,total_orders_scaled,avg_basket_size_scaled,reorder_rate_scaled,items_per_order_scaled,avg_days_between_orders_scaled,dow_entropy_scaled,pct_weekend_orders_scaled,pct_morning_orders_scaled,pct_evening_orders_scaled,n_departments_bought_scaled,pct_organic_scaled,max_order_number_scaled,pct_first_slot_scaled,pct_infrequent_orders_scaled
0,1,-0.335662,-0.690978,1.238152,-0.690978,0.611676,0.040531,-0.250601,1.239643,-0.958027,-1.009504,-0.091477,-0.335662,0.131777,1.064440
1,2,-0.095490,0.678255,0.210583,0.678255,0.003003,0.101082,0.010685,2.346220,-0.958027,0.572875,-0.080579,-0.095490,-0.623873,-0.407397
2,3,-0.215576,-0.446530,0.908586,-0.446530,-0.438904,-0.376735,1.425984,-1.342372,0.347648,-0.482044,0.343204,-0.215576,-0.124456,-0.232236


In [20]:
# Cell 19 — Scale Similarity Vector: MinMaxScaler (range 0–1)
from sklearn.preprocessing import MinMaxScaler

sim_feature_cols = [c for c in SIMILARITY_COLS if c != 'user_id']

scaler_sim = MinMaxScaler()
sim_scaled = scaler_sim.fit_transform(similarity_raw_df[sim_feature_cols].astype('float32'))

similarity_scaled_df = pd.DataFrame(
    sim_scaled,
    columns=[f"{c}_scaled" for c in sim_feature_cols],
)
similarity_scaled_df.insert(0, 'user_id', similarity_raw_df['user_id'].values)

for col in similarity_scaled_df.columns:
    if col != 'user_id':
        similarity_scaled_df[col] = similarity_scaled_df[col].astype('float32')

min_val = similarity_scaled_df.drop('user_id', axis=1).min().min()
max_val = similarity_scaled_df.drop('user_id', axis=1).max().max()
print(f"✓ similarity_scaled_df : {similarity_scaled_df.shape}")
print(f"  min={min_val:.4f} (kỳ vọng 0) | max={max_val:.4f} (kỳ vọng 1)")
similarity_scaled_df.head(3)

✓ similarity_scaled_df : (206209, 14)
  min=0.0000 (kỳ vọng 0) | max=1.0000 (kỳ vọng 1)


,user_id,total_orders_scaled,total_unique_items_scaled,reorder_rate_scaled,avg_basket_size_scaled,preferred_dow_scaled,preferred_hour_scaled,pct_weekend_orders_scaled,top_department_id_scaled,top_aisle_id_scaled,n_departments_bought_scaled,pct_organic_scaled,max_order_number_scaled,pct_first_slot_scaled
0,1,0.072917,0.023448,0.702269,0.070758,0.166667,0.304348,0.300000,0.90,0.571429,0.3,0.254237,0.072917,0.157499
1,2,0.114583,0.139310,0.481970,0.186694,0.166667,0.434783,0.357143,0.75,0.894737,0.6,0.256410,0.114583,0.058391
2,3,0.093750,0.044138,0.631614,0.091456,0.000000,0.695652,0.666667,0.15,0.172932,0.4,0.340909,0.093750,0.123892


---
## PHẦN 7 — Upload 5 files + Ghi manifest

In [21]:
# Cell 20 — Upload 5 files lên Silver
from src.io_utils import upload_dataframe

UPLOAD_TABLES = [
    ("user_features_df",  user_features_df),
    ("eda_stats",         eda_stats_df),
    # stat_report bỏ — p-value/chi2 là phân tích nội bộ, BI không dùng
    ("clustering_scaled", clustering_scaled_df),
    ("similarity_scaled", similarity_scaled_df),
]
manifest_tables = {}

print("Uploading 5 files to Silver...")
for table_name, df in UPLOAD_TABLES:
    upload_dataframe(
        df,
        bucket=config.SILVER_BUCKET,
        prefix=config.SILVER_FEATURES_PREFIX,  # ← đổi
        run_id=RUN_ID,
        table_name=table_name,
    )
    manifest_tables[table_name] = {"rows": len(df), "cols": len(df.columns)}

print("\n✅ Upload xong 5 files:")
for t, info in manifest_tables.items():
    print(f"  ✓ {t:<28} {info['rows']:>10,} rows | {info['cols']} cols")

Uploading 5 files to Silver...
  ✓ uploaded [user_features_df]: 1 file | 206,209 rows
  ✓ uploaded [eda_stats]: 1 file | 18 rows
  ✓ uploaded [clustering_scaled]: 1 file | 206,209 rows
  ✓ uploaded [similarity_scaled]: 1 file | 206,209 rows

✅ Upload xong 5 files:
  ✓ user_features_df                206,209 rows | 25 cols
  ✓ eda_stats                            18 rows | 2 cols
  ✓ clustering_scaled               206,209 rows | 15 cols
  ✓ similarity_scaled               206,209 rows | 14 cols


In [22]:
# Cell 21 — Ghi manifest
from src.supabase_utils import write_manifest

write_manifest(
    bucket=config.SILVER_BUCKET,
    prefix=config.SILVER_FEATURES_PREFIX,  
    run_id=RUN_ID,
    layer="silver_features",
    created_by="bich",
    tables=manifest_tables,
    source_run_id=SILVER_MASTER_RUN_ID,
)

print(f"\n✅ 02b DONE | RUN_ID: {RUN_ID}")
print("→ Copy RUN_ID này vào SILVER_FEATURES_RUN_ID trong Gold layer (03)")

  ✓ manifest → silver-data/instacart/silver-features/20260424T071558Z/_manifest.json

✅ 02b DONE | RUN_ID: 20260424T071558Z
→ Copy RUN_ID này vào SILVER_FEATURES_RUN_ID trong Gold layer (03)


In [23]:
# Cell 22 — Verify: đọc lại từng file để confirm shape khớp
from src.io_utils import load_dataframe

print("Verifying uploads...")
for table_name, df_original in UPLOAD_TABLES:
    df_verify = load_dataframe(
    config.SILVER_BUCKET,
    config.SILVER_FEATURES_PREFIX,  
        RUN_ID,
        table_name,
    )
    assert len(df_verify) == len(df_original), (
        f"❌ {table_name}: verify={len(df_verify):,} ≠ original={len(df_original):,}"
    )
    print(f"  ✓ {table_name:<28} {len(df_verify):>10,} rows — OK")

print("\n✅ Tất cả files verified")

Verifying uploads...
  ✓ loaded [user_features_df]: 1 file | 206,209 rows
  ✓ user_features_df                206,209 rows — OK
  ✓ loaded [eda_stats]: 1 file | 18 rows
  ✓ eda_stats                            18 rows — OK
  ✓ loaded [clustering_scaled]: 1 file | 206,209 rows
  ✓ clustering_scaled               206,209 rows — OK
  ✓ loaded [similarity_scaled]: 1 file | 206,209 rows
  ✓ similarity_scaled               206,209 rows — OK

✅ Tất cả files verified
